# HQ-GAN: Quantum-Enhanced Network Traffic Generation
### Refactored Version with Bug Fixes and Optimizations

**Changes from original:**
- Fixed pickle.dump() bug
- Removed redundant imports and code
- Fixed self-attention reshape bug
- Added proper configuration management
- Improved quantum layer batching
- Made paths configurable for Colab/Local
- Fixed attention dimension handling

## 1. Setup and Installation

In [1]:
# Install required packages (Colab kernel in VS Code)
!pip install torch torchvision torchaudio -q
!pip install pandas pyarrow scikit-learn matplotlib seaborn tqdm -q
!pip install pennylane pennylane-lightning -q

In [2]:
# Mount Google Drive (only needed for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = "/content/drive/MyDrive/supergan/CSE-CIC-IDS22018"
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print("Running locally - ensure data paths are configured correctly")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Data Loading and Preprocessing

In [3]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pickle
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# Configuration
DATA_PATH = "/content/drive/MyDrive/supergan/CSE-CIC-IDS22018"  # Update for your setup
N_SAMPLES = 70000
RANDOM_SEED = 42

# Load parquet files
files = [f for f in os.listdir(DATA_PATH) if f.endswith('.parquet')]
print(f"Found {len(files)} parquet files")

# Load and sample data
df = pq.read_table(os.path.join(DATA_PATH, files[0])).to_pandas()
df = df.sample(n=min(N_SAMPLES, len(df)), random_state=RANDOM_SEED)

print(f"Loaded dataset shape: {df.shape}")
df.head()

Found 10 parquet files
Loaded dataset shape: (70000, 78)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
312779,6,86180965,2,0,0,0.0,0,0,0.000000,0.000000,...,20,0.000000,0.000000,0.0,0.0,86200000.0,0.000000,86200000.0,86200000.0,Benign
339791,6,1668239,8,7,1144,1581.0,677,0,143.000000,227.969925,...,20,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
38167,6,12635,3,4,326,129.0,326,0,108.666664,188.216187,...,20,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,Bot
657214,6,115436126,26,32,1267,25011.0,903,0,48.730770,179.881195,...,20,41821.089844,30929.642578,135058.0,32217.0,10000000.0,48458.574219,10200000.0,9999364.0,Benign
343103,17,27658,2,2,76,180.0,38,38,38.000000,0.000000,...,8,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign


In [4]:
# Data Cleaning and Preprocessing

# Drop missing or infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"After cleaning: {df.shape}")

# Separate features and labels
X = df.drop(columns=['Label'])
y = df['Label']

# Encode labels to binary (Benign=0, Attack=1)
y_binary = np.where(y.str.contains("Benign", case=False, na=False), 0, 1).astype(np.int64)

# Scale features to [0, 1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler for inference
os.makedirs('./models', exist_ok=True)
with open('./models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)  # FIXED: was pickle.dump(scaler, 'wb')
print("✓ Saved scaler to './models/scaler.pkl'")

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Attacks: {y_binary.sum()}, Benign: {len(y_binary) - y_binary.sum()}")

After cleaning: (70000, 78)
✓ Saved scaler to './models/scaler.pkl'
Feature matrix shape: (70000, 77)
Attacks: 13082, Benign: 56918


## 3. Model Definitions

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pennylane as qml
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Architecture hyperparameters
input_dim = X_scaled.shape[1]      # 77 features
latent_dim = 64
num_classes = 2
cond_dim = 16
quantum_dim = 16  # Number of qubits (upgraded from 4 for more expressive quantum bottleneck)
num_heads = 7    # 77 = 7 × 11

Using device: cuda


In [6]:
# Conditional Embedding
class ConditionalEmbedding(nn.Module):
    def __init__(self, num_classes, embed_dim):
        super().__init__()
        self.embed = nn.Embedding(num_classes, embed_dim)
    
    def forward(self, labels):
        return self.embed(labels)


# Quantum Layer - Fixed to properly return tensors

dev = qml.device("lightning.qubit", wires=quantum_dim)

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def quantum_layer_single(x, weights):
    """Process a single sample through the variational quantum circuit."""
    n_layers = weights.shape[0]
    
    # Encode input into quantum state — scale by pi to use full Bloch sphere
    for q in range(quantum_dim):
        qml.RY(x[q] * np.pi, wires=q)
    
    # Apply variational layers
    for layer in range(n_layers):
        # Trainable rotations
        for q in range(quantum_dim):
            qml.RY(weights[layer, q, 0], wires=q)
            qml.RZ(weights[layer, q, 1], wires=q)
        
        # Entanglement layer (circular CNOT pattern)
        for q in range(quantum_dim - 1):
            qml.CNOT(wires=[q, q + 1])
        qml.CNOT(wires=[quantum_dim - 1, 0])
    
    # Return list directly — PennyLane handles this natively with torch interface
    return [qml.expval(qml.PauliZ(i)) for i in range(quantum_dim)]


class QuantumLayer(nn.Module):
    """Wrapper that handles batching for the quantum layer."""
    def __init__(self, quantum_dim, n_layers):
        super().__init__()
        self.quantum_dim = quantum_dim
        self.n_layers = n_layers
        self.weights = nn.Parameter(torch.randn(n_layers, quantum_dim, 2) * 0.1)
    
    def forward(self, x_batch):
        """Process batch by iterating (quantum circuits are inherently sequential)."""
        results = []
        for x in x_batch:
            out = quantum_layer_single(x, self.weights)
            # out is a list of torch scalars — stack into a 1D tensor
            results.append(torch.stack(out))
        return torch.stack(results)

In [7]:
class QuantumEncoder(nn.Module):
    """
    Quantum-encoded generator: latent (64) → encode (16) → quantum (16) → decode (77)
    
    Key design choices:
    - 16 qubits provides 2^16 = 65,536 dimensional Hilbert space
    - Variational layers learn to manipulate quantum state for feature correlation
    - Circular entanglement pattern allows information flow between all qubits
    
    This is a QUANTUM-INSPIRED bottleneck: the quantum circuit forces the network
    to learn compact, correlated representations through its limited measurement outputs.
    """
    def __init__(self, input_dim, quantum_dim, output_dim, n_quantum_layers=2):
        super().__init__()
        self.quantum_dim = quantum_dim
        self.n_quantum_layers = n_quantum_layers
        
        # Encoder: compress to quantum dimension
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, quantum_dim),
            nn.Tanh()  # Match quantum circuit input range [-1, 1]
        )
        
        # Quantum layer as nn.Module
        self.quantum_layer = QuantumLayer(quantum_dim, n_quantum_layers)
        
        # Decoder: expand quantum output to features
        self.decoder = nn.Sequential(
            nn.Linear(quantum_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim),
            nn.Sigmoid()  # Output in [0, 1] to match scaled features
        )
    
    def forward(self, x):
        # Encode to quantum dimension
        z_q = self.encoder(x)  # (batch, 64) → (batch, 16)
        
        # Process through quantum circuit
        q_out = self.quantum_layer(z_q)
        
        # Decode to feature space
        return self.decoder(q_out)

In [8]:
# Multi-Head Self-Attention - FIXED reshape bug
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads=7):
        super().__init__()
        assert dim % num_heads == 0, f"dim {dim} must be divisible by num_heads {num_heads}"
        
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
    
    def forward(self, x):
        batch_size = x.size(0)
        
        # Project: (batch, dim) → (batch, dim)
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        
        # Reshape for multi-head: (batch, num_heads, head_dim)
        Q = Q.view(batch_size, self.num_heads, self.head_dim)
        K = K.view(batch_size, self.num_heads, self.head_dim)
        V = V.view(batch_size, self.num_heads, self.head_dim)
        
        # Attention: (batch, num_heads, head_dim) @ (batch, head_dim, num_heads)
        # This is simplified attention treating heads as sequence
        attn = torch.softmax(Q @ K.transpose(-1, -2) * self.scale, dim=-1)
        out = attn @ V  # (batch, num_heads, head_dim)
        
        # Concatenate: (batch, num_heads * head_dim) = (batch, dim)
        out = out.reshape(batch_size, -1)
        out = self.out_proj(out)
        
        return out + x  # Residual connection

In [9]:
# Generator
class Generator(nn.Module):
    def __init__(self, latent_dim, feature_dim, cond_dim, quantum_dim, n_quantum_layers=2):
        super().__init__()
        
        self.cond_emb = nn.Linear(cond_dim, latent_dim)
        
        # Core: Quantum encoder with variational layers
        self.quantum_encoder = QuantumEncoder(
            input_dim=latent_dim,
            quantum_dim=quantum_dim,
            output_dim=feature_dim,
            n_quantum_layers=n_quantum_layers
        )
        
        # Refinement layers - self-attention helps recover feature correlations
        self.attn1 = MultiHeadSelfAttention(feature_dim, num_heads=7)
        self.attn2 = MultiHeadSelfAttention(feature_dim, num_heads=7)
    
    def forward(self, z, cond):
        # Add conditional embedding
        z = z + self.cond_emb(cond)
        
        # Generate through quantum encoder
        out = self.quantum_encoder(z)
        
        # Refine with attention (helps recover feature correlations lost in bottleneck)
        out = self.attn1(out)
        out = self.attn2(out)
        
        return out

In [10]:
# Discriminator
class Discriminator(nn.Module):
    def __init__(self, feature_dim, cond_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(feature_dim + cond_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )
    
    def forward(self, x, cond):
        return self.model(torch.cat([x, cond], dim=1))

In [11]:
# Gradient Penalty for WGAN-GP
def compute_gradient_penalty(D, real_samples, fake_samples, cond):
    alpha = torch.rand(real_samples.size(0), 1, device=real_samples.device)
    interpolates = alpha * real_samples + ((1 - alpha) * fake_samples)
    interpolates.requires_grad_(True)
    d_interpolates = D(interpolates, cond)
    grad_outputs = torch.ones_like(d_interpolates)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

## 4. Data Preparation and Model Initialization

In [12]:
# Convert to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_binary, dtype=torch.long)

# Split by class for balanced handling
X_attack = X_tensor[y_binary == 1]
y_attack = y_tensor[y_binary == 1]
X_benign = X_tensor[y_binary == 0]
y_benign = y_tensor[y_binary == 0]

def split_data(X, y, train_ratio=0.8, seed=42):
    """Split data with reproducible randomization."""
    perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed))
    n_train = int(len(X) * train_ratio)
    train_idx, test_idx = perm[:n_train], perm[n_train:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# Split each class separately
X_attack_train, X_attack_test, y_attack_train, y_attack_test = split_data(X_attack, y_attack)
X_benign_train, X_benign_test, y_benign_train, y_benign_test = split_data(X_benign, y_benign)

# Combine for training
X_train = torch.cat([X_attack_train, X_benign_train])
y_train = torch.cat([y_attack_train, y_benign_train])
X_test = torch.cat([X_attack_test, X_benign_test])
y_test = torch.cat([y_attack_test, y_benign_test])

# Shuffle training data
perm_train = torch.randperm(len(X_train))
X_train, y_train = X_train[perm_train], y_train[perm_train]

print(f"Training: {len(X_train)} samples (Attack: {(y_train==1).sum()}, Benign: {(y_train==0).sum()})")
print(f"Test: {len(X_test)} samples")

Training: 55999 samples (Attack: 10465, Benign: 45534)
Test: 14001 samples


In [13]:
# Initialize models
cond_embed = ConditionalEmbedding(num_classes, cond_dim).to(device)
G = Generator(latent_dim, input_dim, cond_dim, quantum_dim).to(device)
D = Discriminator(input_dim, cond_dim).to(device)

# Optimizers
lr = 1e-4
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.9))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.9))

# DataLoader
batch_size = 64
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"✓ Models initialized. G params: {sum(p.numel() for p in G.parameters()):,}, D params: {sum(p.numel() for p in D.parameters()):,}")

✓ Models initialized. G params: 63,629, D params: 12,161


## 5. Training Loop

In [16]:
# Training Configuration
epochs = 12
lambda_gp = 10
n_critic = 5

training_history = {'G_loss': [], 'D_loss': []}

for epoch in range(epochs):
    G.train()
    D.train()
    
    epoch_g_losses = []
    epoch_d_losses = []
    
    for batch_idx, (real_samples, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)):
        real_samples = real_samples.to(device)
        labels = labels.to(device)
        cond = cond_embed(labels)
        
        # Train Discriminator (n_critic times)
        for _ in range(n_critic):
            z_D = torch.randn(real_samples.size(0), latent_dim, device=device)
            fake_samples_D = G(z_D, cond).detach()
            
            # Detach real_samples and cond from any previous graph
            real_detached = real_samples.detach()
            cond_detached = cond.detach()

            D_real = D(real_detached, cond_detached)
            D_fake = D(fake_samples_D, cond_detached)
            gp = compute_gradient_penalty(D, real_detached, fake_samples_D, cond_detached)
            
            loss_D = -torch.mean(D_real) + torch.mean(D_fake) + lambda_gp * gp
            
            optimizer_D.zero_grad()
            loss_D.backward()
            optimizer_D.step()
        
        # Train Generator
        z_G = torch.randn(real_samples.size(0), latent_dim, device=device)
        fake_samples_G = G(z_G, cond)
        D_fake = D(fake_samples_G, cond)
        loss_G = -torch.mean(D_fake)
        
        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()
        
        epoch_g_losses.append(loss_G.item())
        epoch_d_losses.append(loss_D.item())
    
    avg_g_loss = np.mean(epoch_g_losses)
    avg_d_loss = np.mean(epoch_d_losses)
    training_history['G_loss'].append(avg_g_loss)
    training_history['D_loss'].append(avg_d_loss)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {avg_d_loss:.4f} | Loss G: {avg_g_loss:.4f}", flush=True)

print("\n✓ GAN Training Complete!")

Epoch 1/12:   0%|          | 0/875 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(training_history['D_loss'], label='Discriminator Loss')
plt.plot(training_history['G_loss'], label='Generator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Loss Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Save Trained Models

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/supergan/checkpoints"
LOCAL_CHECKPOINT_DIR = "./checkpoints"

# Create checkpoint directory
checkpoint_dir = CHECKPOINT_DIR if IS_COLAB else LOCAL_CHECKPOINT_DIR
os.makedirs(checkpoint_dir, exist_ok=True)

# Save trained models
torch.save({
    'G_state_dict': G.state_dict(),
    'D_state_dict': D.state_dict(),
    'cond_embed_state_dict': cond_embed.state_dict(),
    'config': {
        'latent_dim': latent_dim,
        'input_dim': input_dim,
        'cond_dim': cond_dim,
        'quantum_dim': quantum_dim,
        'num_classes': num_classes
    }
}, os.path.join(checkpoint_dir, 'trained_gan_model.pth'))

print(f"✓ Saved GAN model to {os.path.join(checkpoint_dir, 'trained_gan_model.pth')}")

## 7. Generate Synthetic Data

In [ ]:
G.eval()

N_total = 40000
N_per_class = N_total // 2

# Get real sample indices by class
benign_idx = (y_train == 0).nonzero(as_tuple=True)[0]
attack_idx = (y_train == 1).nonzero(as_tuple=True)[0]

# Select real samples (up to N_per_class)
real_benign = min(len(benign_idx), N_per_class)
real_attack = min(len(attack_idx), N_per_class)

benign_real_idx = benign_idx[torch.randperm(len(benign_idx))[:real_benign]]
attack_real_idx = attack_idx[torch.randperm(len(attack_idx))[:real_attack]]

X_benign_real = X_train[benign_real_idx]
y_benign_real = y_train[benign_real_idx]
X_attack_real = X_train[attack_real_idx]
y_attack_real = y_train[attack_real_idx]

# Calculate synthetic samples needed
n_syn_benign = N_per_class - real_benign
n_syn_attack = N_per_class - real_attack

print(f"Generating synthetic data:")
print(f"  Benign: {n_syn_benign} synthetic + {real_benign} real")
print(f"  Attack: {n_syn_attack} synthetic + {real_attack} real")

In [ ]:
# Generate synthetic samples
with torch.no_grad():
    # Benign synthetic
    if n_syn_benign > 0:
        z_benign = torch.randn(n_syn_benign, latent_dim, device=device)
        c_benign = cond_embed(torch.zeros(n_syn_benign, dtype=torch.long, device=device))
        X_benign_synth = G(z_benign, c_benign).cpu()
        y_benign_synth = torch.zeros(n_syn_benign, dtype=y_train.dtype)
    else:
        X_benign_synth = torch.empty(0, X_train.shape[1])
        y_benign_synth = torch.empty(0, dtype=y_train.dtype)
    
    # Attack synthetic
    if n_syn_attack > 0:
        z_attack = torch.randn(n_syn_attack, latent_dim, device=device)
        c_attack = cond_embed(torch.ones(n_syn_attack, dtype=torch.long, device=device))
        X_attack_synth = G(z_attack, c_attack).cpu()
        y_attack_synth = torch.ones(n_syn_attack, dtype=y_train.dtype)
    else:
        X_attack_synth = torch.empty(0, X_train.shape[1])
        y_attack_synth = torch.empty(0, dtype=y_train.dtype)

# Combine real and synthetic
X_benign_final = torch.cat([X_benign_real, X_benign_synth])
y_benign_final = torch.cat([y_benign_real, y_benign_synth])
X_attack_final = torch.cat([X_attack_real, X_attack_synth])
y_attack_final = torch.cat([y_attack_real, y_attack_synth])

X_train_balanced = torch.cat([X_benign_final, X_attack_final])
y_train_balanced = torch.cat([y_benign_final, y_attack_final])

# Shuffle balanced dataset
perm_bal = torch.randperm(X_train_balanced.size(0))
X_train_balanced = X_train_balanced[perm_bal]
y_train_balanced = y_train_balanced[perm_bal]

print(f"\n✓ Balanced training set: {len(X_train_balanced)} samples")
print(f"  Benign: {(y_train_balanced == 0).sum().item()}")
print(f"  Attack: {(y_train_balanced == 1).sum().item()}")

## 8. Train Classifier on Balanced Data

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.net(x).squeeze(1)

clf = SimpleClassifier(input_dim).to(device)

# Class-weighted loss for imbalanced data
num_pos = (y_train_balanced == 1).sum().item()
num_neg = (y_train_balanced == 0).sum().item()
pos_weight = torch.tensor([num_neg / max(1, num_pos)], dtype=torch.float32, device=device)
clf_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
clf_opt = optim.Adam(clf.parameters(), lr=1e-4, betas=(0.9, 0.999))

In [ ]:
# Train classifier
clf_epochs = 5
clf_train_loader = DataLoader(
    TensorDataset(X_train_balanced, y_train_balanced.float()),
    batch_size=64,
    shuffle=True
)

for ep in range(clf_epochs):
    clf.train()
    total_loss = 0.0
    for Xb, yb in clf_train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = clf(Xb)
        loss = clf_criterion(logits, yb)
        clf_opt.zero_grad()
        loss.backward()
        clf_opt.step()
        total_loss += loss.item() * Xb.size(0)
    print(f"[Classifier] Epoch {ep+1}/{clf_epochs} | Loss: {total_loss/len(X_train_balanced):.4f}")

print("✓ Classifier training complete!")

## 9. Evaluate Classifier

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

clf.eval()
y_true, y_scores = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = clf(Xb)
        y_true.append(yb.cpu().numpy())
        y_scores.append(torch.sigmoid(logits).cpu().numpy())

y_true = np.concatenate(y_true)
y_scores = np.concatenate(y_scores)

# Adaptive threshold using Youden's J statistic
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
J = tpr - fpr
best_thresh = thresholds[np.argmax(J)]

y_pred = (y_scores >= best_thresh).astype(int)

print(f"Adaptive threshold: {best_thresh:.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_true, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_true, y_pred, zero_division=0)}")

In [ ]:
# Plot ROC and PR curves
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC Curve
roc_auc = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
axes[0].set_xlim([-0.01, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('Receiver Operating Characteristic')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# PR Curve
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)
axes[1].plot(recall, precision, color='blue', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
axes[1].set_xlim([0.0, 1.05])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Save Complete Checkpoint

In [ ]:
checkpoint_path = os.path.join(checkpoint_dir, 'FULL_CHECKPOINT.pth')
torch.save({
    'G_state_dict': G.state_dict(),
    'D_state_dict': D.state_dict(),
    'cond_embed_state_dict': cond_embed.state_dict(),
    'clf_state_dict': clf.state_dict(),
    'X_train_balanced': X_train_balanced,
    'y_train_balanced': y_train_balanced,
    'X_test': X_test,
    'y_test': y_test,
    'input_dim': input_dim,
    'latent_dim': latent_dim,
    'best_thresh': best_thresh,
    'accuracy': accuracy_score(y_true, y_pred),
    'training_history': training_history
}, checkpoint_path)

print(f"✓ Checkpoint saved: {checkpoint_path}")
print(f"  Accuracy: {accuracy_score(y_true, y_pred):.4f}")

## 11. Load Checkpoint (For Inference)

In [ ]:
# Load checkpoint and use trained models
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Reconstruct models
G.load_state_dict(checkpoint['G_state_dict'])
D.load_state_dict(checkpoint['D_state_dict'])
clf.load_state_dict(checkpoint['clf_state_dict'])
cond_embed.load_state_dict(checkpoint['cond_embed_state_dict'])

# Load data
X_test = checkpoint['X_test']
y_test = checkpoint['y_test']
best_thresh = checkpoint['best_thresh']

print(f"✓ Loaded checkpoint with accuracy: {checkpoint['accuracy']:.4f}")

# Usage:
# - Generate: G(torch.randn(100, latent_dim, device=device), cond_embed(labels))
# - Classify: clf(X_new)

## 12. Generate Synthetic Samples (Post-Training)

In [ ]:
# ============================================================
# 12. Generate AND EXPORT Synthetic Samples (Post-Training)
# ============================================================
G.eval()

# Load original feature names for proper CSV export
import json

# Get feature names from original dataset
files = [f for f in os.listdir(DATA_PATH) if f.endswith('.parquet')]
df_sample = pq.read_table(os.path.join(DATA_PATH, files[0])).to_pandas()
feature_names = df_sample.drop(columns=['Label']).columns.tolist()

print(f"Feature names extracted: {len(feature_names)} features")

# Generate large batch of synthetic samples
N_SYNTH_ATTACK = 5000   # Synthetic attack samples (rare class)
N_SYNTH_BENIGN = 5000   # Synthetic benign samples

print(f"\nGenerating {N_SYNTH_ATTACK} attack and {N_SYNTH_BENIGN} benign synthetic samples...")

with torch.no_grad():
    # Generate attack synthetic samples
    z_attack = torch.randn(N_SYNTH_ATTACK, latent_dim, device=device)
    c_attack = cond_embed(torch.ones(N_SYNTH_ATTACK, dtype=torch.long, device=device))
    X_synth_attack = G(z_attack, c_attack).cpu().numpy()

    # Generate benign synthetic samples
    z_benign = torch.randn(N_SYNTH_BENIGN, latent_dim, device=device)
    c_benign = cond_embed(torch.zeros(N_SYNTH_BENIGN, dtype=torch.long, device=device))
    X_synth_benign = G(z_benign, c_benign).cpu().numpy()

# Create DataFrames with proper column names
df_synth_attack = pd.DataFrame(X_synth_attack, columns=feature_names)
df_synth_attack['Label'] = 'Attack'

df_synth_benign = pd.DataFrame(X_synth_benign, columns=feature_names)
df_synth_benign['Label'] = 'Benign'

# Export to CSV
os.makedirs('./synthetic_samples', exist_ok=True)

attack_path = './synthetic_samples/synthetic_attacks.csv'
benign_path = './synthetic_samples/synthetic_benign.csv'
combined_path = './synthetic_samples/synthetic_samples_combined.csv'

df_synth_attack.to_csv(attack_path, index=False)
df_synth_benign.to_csv(benign_path, index=False)
pd.concat([df_synth_attack, df_synth_benign]).to_csv(combined_path, index=False)

# Also save in parquet format (more efficient)
df_synth_attack.to_parquet(attack_path.replace('.csv', '.parquet'), index=False)
df_synth_benign.to_parquet(benign_path.replace('.csv', '.parquet'), index=False)

print(f"\n✓ EXPORTED SYNTHETIC SAMPLES:")
print(f"  Attacks: {attack_path} ({len(df_synth_attack)} samples)")
print(f"  Benign:  {benign_path} ({len(df_synth_benign)} samples)")
print(f"  Combined: {combined_path}")
print(f"\n✓ Also saved in Parquet format in ./synthetic_samples/")

# Quality check: Show statistics
print(f"\n=== Synthetic Data Statistics ===")
print(f"Attack samples: {df_synth_attack.shape}, Value range: [{df_synth_attack.iloc[:, :-1].values.min():.4f}, {df_synth_attack.iloc[:, :-1].values.max():.4f}]")
print(f"Benign samples: {df_synth_benign.shape}, Value range: [{df_synth_benign.iloc[:, :-1].values.min():.4f}, {df_synth_benign.iloc[:, :-1].values.max():.4f}]")

# Show sample rows
print(f"\n=== Sample Synthetic Attack (first 3 rows) ===")
display(df_synth_attack.head(3))